In [0]:
%pip install --upgrade typing_extensions transformers torch

Python interpreter will be restarted.
  Using cached typing_extensions-4.14.0-py3-none-any.whl (43 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.1.1
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-feae712a-14a8-4993-8229-164ef850a778
    Can't uninstall 'typing-extensions'. No files were found to uninstall.
Python interpreter will be restarted.


In [0]:
from pyspark.sql import SparkSession
from transformers import pipeline
from pyspark.sql.functions import to_date, from_unixtime, col, first, last, max, min, avg, sum, count, when, udf, date_format, hour, expr, minute, floor, concat, lpad, lit, abs, mean, stddev
from pyspark.sql.functions import monotonically_increasing_id, coalesce, pandas_udf
from pyspark.sql.types import StructType, StructField, StringType, FloatType

In [0]:
%sql
SELECT * FROM `nvidia_data` LIMIT 5

Date,Close_NVDA,High_NVDA,Low_NVDA,Open_NVDA,Volume_NVDA,Period_NVDA,Shares_Out_NVDA,Market_Cap_NVDA,Trailing_PE_NVDA,Forward_PE_NVDA,Total_Revenue_NVDA,PS_Ratio_NVDA,Total_Debt_NVDA,Cash_Equivalents_NVDA,TD_CE_Missing_NVDA,Shares_Out_Missing_NVDA,PE_Missing_NVDA,PS_Missing_NVDA,EV_NVDA
1999-01-22,0.0376117043197155,0.0447753115840147,0.0355814649774072,0.040118782563947,2714688000,before_trump,null,null,0.012793096707386225,0.009129054446532888,130497003520,null,0.0,0.0,1,1,0,1,null
1999-01-25,0.0415520668029785,0.0420289058282125,0.0376117140461148,0.0405965508934446,510480000,before_trump,null,null,0.014133356055434865,0.010085453107519054,130497003520,null,0.0,0.0,1,1,0,1,null
1999-01-26,0.0383278876543045,0.0428652059573384,0.0377309182952898,0.042028901596367,343200000,before_trump,null,null,0.013036696481055952,0.009302885352986527,130497003520,null,0.0,0.0,1,1,0,1,null
1999-01-27,0.0382086783647537,0.0394026137342712,0.0362976500460264,0.0384470978668602,244368000,before_trump,null,null,0.012996149103657722,0.009273951059406238,130497003520,null,0.0,0.0,1,1,0,1,null
1999-01-28,0.0380885563790798,0.0384471029006281,0.0378501368457577,0.038208683367306,227520000,before_trump,null,null,0.012955291285401292,0.009244795237640727,130497003520,null,0.0,0.0,1,1,0,1,null


In [0]:
%sql
SELECT COUNT(*) FROM `nvidia_data`

count(1)
6586


In [0]:
%sql
SELECT * FROM `amd_news` LIMIT 5

category,datetime,headline,id,image,related,source,summary,url
company,1718121435,US Weighs More Limits on China’s Access to Chips Needed for AI,128201039,https://s.yimg.com/ny/api/res/1.2/zcF2AjYxPNAZTe3vV_MPjQ--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD04MDA-/https://media.zenfs.com/en/bloomberg_technology_68/71df878c23a711e3667b743beb3a243e,AMD,Yahoo,"(Bloomberg) -- The Biden administration is considering further restrictions on China’s access to chip technology used for artificial intelligence, targeting new hardware that’s only now making its way into the market, people familiar with the matter said.Most Read from BloombergMusk to Ban Apple Devices If OpenAI Is Integrated Into OSApple Hits Record After Unveiling ‘AI for the Rest of Us’ PlanNYC Landlord to Sell Office Building at Roughly 67% DiscountRussia Is Sending Young Africans to Die in",https://finnhub.io/api/news?id=8529124c6fefcaca7359315ed5a29562317eb08be895841c92dabb65d7cee5cc
company,1718125713,"Advanced Micro Devices, Inc. (NASDAQ:AMD) Downgraded by Morgan Stanley",128202969,https://media.zenfs.com/en/insidermonkey.com/b088bb723fa58ecbe0a85c8080e57fcd,AMD,Yahoo,"We recently compiled the list of the Analysts on Wall Street Lower Ratings for These 10 Stocks. In this article, we are going to take a look at where Advanced Micro Devices, Inc. (NASDAQ:AMD) stands against the other stocks that received a downgrade from Wall Street analysts. But first, we are going to take a look […]",https://finnhub.io/api/news?id=dd19be4b6a03b13f61be8034483ab84b2ff0b3708db601cee3a5d8fa5dbe5d47
company,1718124218,"Nvidia: Too Expensive, Competition Is Rising, And Earnings Growth Is Slowing",128206498,https://static.seekingalpha.com/cdn/s3/uploads/getty_images/1412721464/image_1412721464.jpg?io=getty-c-w1536,AMD,SeekingAlpha,"Nvidia's stock rally has intensified, driven by investor enthusiasm about the AI boom. Raad whyÂ investors' expectations are far too great for NVDA stock.",https://finnhub.io/api/news?id=1e73d2c723c5a10e41e0fb9ed125ed58877cb7036c62a137bbc7e0f4814feecc
company,1718123144,Is AMD Stock a Buy After Following Nvidia's Footsteps?,128202970,https://g.foolcdn.com/editorial/images/780260/amd-headquarters-santa-clara-with-amd-logo-on-building_amd_advance.jpg,AMD,Yahoo,"Advanced Micro Devices is releasing new AI chips annually, following the annual cadence Nvidia announced late last year.",https://finnhub.io/api/news?id=f3e933f744403a1966b998d4d6a3959e9cc58259edd122ccec3f0d919343fde5
company,1718118000,"Nvidia: Great Outlook Remains, But Stock Has Had Enough For Now (Rating Downgrade)",128203930,https://static.seekingalpha.com/cdn/s3/uploads/getty_images/2094940552/image_2094940552.jpg?io=getty-c-w1536,AMD,SeekingAlpha,NvidiaÂ Corporation is firing on all cylinders as it continues to dominate in the AI accelerator sector. Find out more on NVDA stock here.,https://finnhub.io/api/news?id=fecd913ef24973ed6357961c3f6a6b3fb2d1d9780406a4f8068b623571a5027d


In [0]:
%sql
SELECT COUNT(*) FROM `amd_news`

count(1)
4515


In [0]:
df_prices = spark.sql("SELECT * FROM nvidia_data")

In [0]:
df_prices.columns

Out[7]: ['Date',
 'Close_NVDA',
 'High_NVDA',
 'Low_NVDA',
 'Open_NVDA',
 'Volume_NVDA',
 'Period_NVDA',
 'Shares_Out_NVDA',
 'Market_Cap_NVDA',
 'Trailing_PE_NVDA',
 'Forward_PE_NVDA',
 'Total_Revenue_NVDA',
 'PS_Ratio_NVDA',
 'Total_Debt_NVDA',
 'Cash_Equivalents_NVDA',
 'TD_CE_Missing_NVDA',
 'Shares_Out_Missing_NVDA',
 'PE_Missing_NVDA',
 'PS_Missing_NVDA',
 'EV_NVDA']

In [0]:
df_prices = df_prices.drop('Period_NVDA','Shares_Out_NVDA','Market_Cap_NVDA','Trailing_PE_NVDA','Forward_PE_NVDA','Total_Revenue_NVDA','PS_Ratio_NVDA','Total_Debt_NVDA','Cash_Equivalents_NVDA','TD_CE_Missing_NVDA','Shares_Out_Missing_NVDA','PE_Missing_NVDA','PS_Missing_NVDA','EV_NVDA')
display(df_prices.limit(5))

Date,Close_NVDA,High_NVDA,Low_NVDA,Open_NVDA,Volume_NVDA
1999-01-22,0.0376117043197155,0.0447753115840147,0.0355814649774072,0.040118782563947,2714688000
1999-01-25,0.0415520668029785,0.0420289058282125,0.0376117140461148,0.0405965508934446,510480000
1999-01-26,0.0383278876543045,0.0428652059573384,0.0377309182952898,0.042028901596367,343200000
1999-01-27,0.0382086783647537,0.0394026137342712,0.0362976500460264,0.0384470978668602,244368000
1999-01-28,0.0380885563790798,0.0384471029006281,0.0378501368457577,0.038208683367306,227520000


In [0]:
df_prices = df_prices.filter(df_prices["Date"] >= lit("2024-06-10"))
display(df_prices)

Date,Close_NVDA,High_NVDA,Low_NVDA,Open_NVDA,Volume_NVDA
2024-06-10,121.74999237060548,123.05955958930453,116.9715638416941,120.33046067723372,314162700
2024-06-11,120.88021087646484,122.83972700807324,118.7107397776175,121.739991951163,222551200
2024-06-12,125.16915130615234,126.84873770805828,122.53980200765416,123.02967915018486,299595000
2024-06-13,129.5780792236328,129.7680348697194,127.128685680218,129.35813218658845,260704500
2024-06-14,131.84751892089844,132.80727390157034,128.28839829543193,129.92799370452428,309320400
2024-06-17,130.94773864746094,133.69706139159663,129.54808953427707,132.9572533982128,288504400
2024-06-18,135.54660034179688,136.29641557179684,130.6578056523983,131.10769173939232,294335100
2024-06-20,130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400
2024-06-21,126.53881072998048,130.5978154647857,124.26937345205326,127.08867825178736,655484700
2024-06-24,118.08090209960938,124.42933613805307,118.01091965046172,123.20963548646723,476060900


In [0]:
# df_prices.write.mode("overwrite").option("header", True).csv("dbfs:/FileStore/")

In [0]:
df_news = spark.sql("SELECT * FROM amd_news")

In [0]:
df_news = df_news.withColumn("DateTime", from_unixtime(col("datetime")))
df_news = df_news.drop('category', 'id', 'image', 'source', 'url', 'related') # remove related when multiple tickers being used
display(df_news.limit(5))


DateTime,headline,summary
2024-06-11 15:57:15,US Weighs More Limits on China’s Access to Chips Needed for AI,"(Bloomberg) -- The Biden administration is considering further restrictions on China’s access to chip technology used for artificial intelligence, targeting new hardware that’s only now making its way into the market, people familiar with the matter said.Most Read from BloombergMusk to Ban Apple Devices If OpenAI Is Integrated Into OSApple Hits Record After Unveiling ‘AI for the Rest of Us’ PlanNYC Landlord to Sell Office Building at Roughly 67% DiscountRussia Is Sending Young Africans to Die in"
2024-06-11 17:08:33,"Advanced Micro Devices, Inc. (NASDAQ:AMD) Downgraded by Morgan Stanley","We recently compiled the list of the Analysts on Wall Street Lower Ratings for These 10 Stocks. In this article, we are going to take a look at where Advanced Micro Devices, Inc. (NASDAQ:AMD) stands against the other stocks that received a downgrade from Wall Street analysts. But first, we are going to take a look […]"
2024-06-11 16:43:38,"Nvidia: Too Expensive, Competition Is Rising, And Earnings Growth Is Slowing","Nvidia's stock rally has intensified, driven by investor enthusiasm about the AI boom. Raad whyÂ investors' expectations are far too great for NVDA stock."
2024-06-11 16:25:44,Is AMD Stock a Buy After Following Nvidia's Footsteps?,"Advanced Micro Devices is releasing new AI chips annually, following the annual cadence Nvidia announced late last year."
2024-06-11 15:00:00,"Nvidia: Great Outlook Remains, But Stock Has Had Enough For Now (Rating Downgrade)",NvidiaÂ Corporation is firing on all cylinders as it continues to dominate in the AI accelerator sector. Find out more on NVDA stock here.


In [0]:
# convert datetime format to readable date
df_news = df_news.withColumn("Date", to_date(col("DateTime")))
df_news = df_news.drop('DateTime')
display(df_news.limit(5))

headline,summary,Date
US Weighs More Limits on China’s Access to Chips Needed for AI,"(Bloomberg) -- The Biden administration is considering further restrictions on China’s access to chip technology used for artificial intelligence, targeting new hardware that’s only now making its way into the market, people familiar with the matter said.Most Read from BloombergMusk to Ban Apple Devices If OpenAI Is Integrated Into OSApple Hits Record After Unveiling ‘AI for the Rest of Us’ PlanNYC Landlord to Sell Office Building at Roughly 67% DiscountRussia Is Sending Young Africans to Die in",2024-06-11
"Advanced Micro Devices, Inc. (NASDAQ:AMD) Downgraded by Morgan Stanley","We recently compiled the list of the Analysts on Wall Street Lower Ratings for These 10 Stocks. In this article, we are going to take a look at where Advanced Micro Devices, Inc. (NASDAQ:AMD) stands against the other stocks that received a downgrade from Wall Street analysts. But first, we are going to take a look […]",2024-06-11
"Nvidia: Too Expensive, Competition Is Rising, And Earnings Growth Is Slowing","Nvidia's stock rally has intensified, driven by investor enthusiasm about the AI boom. Raad whyÂ investors' expectations are far too great for NVDA stock.",2024-06-11
Is AMD Stock a Buy After Following Nvidia's Footsteps?,"Advanced Micro Devices is releasing new AI chips annually, following the annual cadence Nvidia announced late last year.",2024-06-11
"Nvidia: Great Outlook Remains, But Stock Has Had Enough For Now (Rating Downgrade)",NvidiaÂ Corporation is firing on all cylinders as it continues to dominate in the AI accelerator sector. Find out more on NVDA stock here.,2024-06-11


In [0]:
# show how many news articles for the same date
duplicates = df_news.groupBy("Date").agg(count("*").alias("count")).filter("count > 1")
display(duplicates)

Date,count
2024-09-18,12
2025-02-16,2
2024-06-12,22
2024-08-27,15
2024-10-24,11
2024-11-02,5
2024-11-25,9
2025-02-01,4
2025-03-23,2
2025-04-17,33


In [0]:
df_prices.count()

Out[15]: 200

In [0]:
df_prices.columns

Out[16]: ['Date', 'Close_NVDA', 'High_NVDA', 'Low_NVDA', 'Open_NVDA', 'Volume_NVDA']

In [0]:
# Add relative gap column
df_with_gap = df_prices.withColumn("Daily_Gap", abs(col("Open_NVDA") - col("Close_NVDA")) / col("Open_NVDA"))

# Calculate mean and stddev
gap_stats = df_with_gap.select(
    mean("Daily_Gap").alias("Mean_Gap"),
    stddev("Daily_Gap").alias("Stddev_Gap")
).collect()[0]

mean_gap = gap_stats["Mean_Gap"]
stddev_gap = gap_stats["Stddev_Gap"]
threshold = mean_gap + 2 * stddev_gap

# Filter abnormally large gaps
df_outliers = df_with_gap.filter(col("Daily_Gap") > threshold)


In [0]:
print(threshold)

0.05965991499133652


In [0]:
# Ensure column is in date format
df_outliers = df_outliers.withColumn("Date", to_date("Date"))
df_outliers.count() # - only 16 outliers -- only 16 events will get tagged
# Filter rows to only June 2024 - May 2025 (as that is all our data includes)
# df_outliers = df_outliers.filter(df_outliers["Date"] >= lit("2024-06-01"))

Out[19]: 11

In [0]:
df_news.count()

Out[20]: 4515

In [0]:
display(df_news.filter(col("summary").isNull()))

headline,summary,Date
Hold Nvidia: Better Opportunities Await,null,2024-06-21
My Top 3 Stock Picks For 2024: Mid-Year Update,null,2024-06-23
AMD And Its Real Value,null,2024-06-24
Intel: Between A Rock (NVIDIA) And A Hard Place (AMD),null,2024-07-04
Wall Street Breakfast: What Moved Markets,null,2024-07-06
AI PC Stocks: Emerging 2024 And 2025 Story,null,2024-07-08
The Rapidly Evolving World Of AI PCs,null,2024-07-18
"Google: The More It Drops, The More I'll Buy",null,2024-07-25
Tracking Baillie Gifford's 13F Portfolio - Q2 2024 Update,null,2024-07-28
Wall Street Lunch:The Return Of The Dual Mandate,null,2024-07-31


In [0]:
# remove any rows with null in it
df_news = df_news.filter(
    (col("summary").isNotNull()) & (col("summary") != "null")
)
df_news.count()

Out[22]: 4432

In [0]:
# remove markinting news articles
df_news = df_news.filter(~col("summary").contains("Zacks.com"))
df_news = df_news.filter(~col("summary").contains("market.com"))
df_news.count()

Out[23]: 3303

In [0]:
# remove summaries that dont mention nvda
df_news = df_news.filter(
    col("summary").rlike("(?i)\\b(amd)\\b") |
    col("headline").rlike("(?i)\\b(amd)\\b")
)
df_news.count()

Out[24]: 1559

In [0]:
# join the news with the price outliers on date
df_tagged_events = df_news.join(
    df_outliers,
    on=["Date"],
    how="inner"
)
df_tagged_events.count()

Out[25]: 70

In [0]:
df_outliers.columns

Out[26]: ['Date',
 'Close_NVDA',
 'High_NVDA',
 'Low_NVDA',
 'Open_NVDA',
 'Volume_NVDA',
 'Daily_Gap']

In [0]:
display(df_tagged_events)

Date,headline,summary,Close_NVDA,High_NVDA,Low_NVDA,Open_NVDA,Volume_NVDA,Daily_Gap
2024-06-20,"Housing trends, HPE CEO on Nvidia partnership: Market Domination","There's never enough time in the day to trade, as Market Domination Hosts Julie Hyman and Josh Lipton walk investors through the final trading hour of Thursday, June 20. They cover the top trending stocks and market movements ahead of the closing bell. Hewlett Packard Enterprise (HPE) CEO Antonio Neri discusses HPE's new partnership with Nvidia (NVDA) on its line of ""Nvidia AI Computing by HPE"" product offerings. National Association of Home Builders (NAHB) CEO Jim Tobin stops into the studio to tackle some of the biggest challenges the US housing market is currently facing. Yahoo Finance's top trending stock tickers this hour include Gilead Sciences (GILD), commercial-grade EV maker Nikola (NKLA), and Advanced Micro Devices (AMD). This post was written by Luke Carberry Mogan.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693
2024-06-20,AMD named top pick at Piper Sandler,"Shares of Advanced Micro Devices (AMD) popped on Thursday. Piper Sandler analyst Harsh Kumar named the stock as a top pick in the large-cap space. He cites a number of factors, including the competitive positioning of its MI products, and says that he sees AMD ""as having bright prospects moving into the back half of the year."" Kumar has an Overweight rating and $175 price target on the stock. Yahoo Finance's Josh Lipton and Julie Hyman discuss the call in the video above. For more expert insight and the latest market action, click here to watch this full episode of Market Domination. This post was written by Stephanie Mikulich.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693
2024-06-20,AMD Shares Now Top Piper Pick Because of AI Server Chips,Shares of Advanced Micro Devices traded higher Thursday after a Pipe Sandler analyst issued a bullish note on the artificial intelligence business of the chip maker.,130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693
2024-06-20,Why Are AMD (AMD) Shares Soaring Today,"Shares of computer processor maker AMD (NASDAQ:AMD) jumped 7.5% in the morning session after Piper Sandler analyst Harsh Kumar named the company a ""Top Pick"" and maintained an Overweight (Buy) rating on the stock.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693
2024-06-20,AMD Stock Is Set Up For Another Spike,AMD's upcoming quarterly earnings report on August 6th could serve as a catalyst for a potential spike in stock price. Find out if AMD stock is a buy.,130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693
2024-06-20,Piper Sandler names AMD as top large-cap pick into 2H24,"Piper Sandler analysts named AMD (NASDAQ: NASDAQ:AMD) their top large-cap pick for the second half of 2024, citing positive feedback from their discussions with the chipmaker's management in Europe last week.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693
2024-06-20,"AMD Hack Won’t Have a Material Impact on Business, Company Says","(Bloomberg) -- Advanced Micro Devices Inc. found hackers made off with limited information during a recent cyberattack, saying the infiltration shouldn’t have significant impact on its operations.Most Read from BloombergCar Dealerships Across US Halt Services After CyberattackPutin’s Hybrid War Opens a Second Front on NATO’s Eastern BorderHedge Fund Talent Schools Are Looking for the Perfect TraderCar Dealers Across US Are Crippled by a Second CyberattackWhat to Know About the Deadly Flesh-Eatin",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693
2024-07-11,Analysts revamp AMD stock price target on AI deal,Thi

In [0]:
def analyze_sentiment(text):
    if not text:
        return ("NEUTRAL", 0.0)
    
    from transformers import pipeline
    sentiment_pipeline = pipeline("sentiment-analysis", model="yiyanghkust/finbert-tone")
    
    # Call correctly
    result = sentiment_pipeline(text)[0]  # ✅ CORRECT CALL
    return (result["label"], float(result["score"]))

# Register UDF with schema
schema = StructType([
    StructField("label", StringType(), True),
    StructField("score", FloatType(), True)
])

sentiment_udf = udf(analyze_sentiment, schema)

In [0]:
df_with_sentiment = df_tagged_events.withColumn("sentiment_struct", sentiment_udf("summary"))
df_with_sentiment = df_with_sentiment.withColumn("sentiment", df_with_sentiment["sentiment_struct.label"])
df_with_sentiment = df_with_sentiment.withColumn("sentiment_score", df_with_sentiment["sentiment_struct.score"])
df_with_sentiment = df_with_sentiment.drop("sentiment_struct")


In [0]:
df_with_sentiment.columns

Out[30]: ['Date',
 'headline',
 'summary',
 'Close_NVDA',
 'High_NVDA',
 'Low_NVDA',
 'Open_NVDA',
 'Volume_NVDA',
 'Daily_Gap',
 'sentiment',
 'sentiment_score']

In [0]:
display(df_with_sentiment)

Date,headline,summary,Close_NVDA,High_NVDA,Low_NVDA,Open_NVDA,Volume_NVDA,Daily_Gap,sentiment,sentiment_score
2024-06-20,"Housing trends, HPE CEO on Nvidia partnership: Market Domination","There's never enough time in the day to trade, as Market Domination Hosts Julie Hyman and Josh Lipton walk investors through the final trading hour of Thursday, June 20. They cover the top trending stocks and market movements ahead of the closing bell. Hewlett Packard Enterprise (HPE) CEO Antonio Neri discusses HPE's new partnership with Nvidia (NVDA) on its line of ""Nvidia AI Computing by HPE"" product offerings. National Association of Home Builders (NAHB) CEO Jim Tobin stops into the studio to tackle some of the biggest challenges the US housing market is currently facing. Yahoo Finance's top trending stock tickers this hour include Gilead Sciences (GILD), commercial-grade EV maker Nikola (NKLA), and Advanced Micro Devices (AMD). This post was written by Luke Carberry Mogan.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Neutral,0.9999926
2024-06-20,AMD named top pick at Piper Sandler,"Shares of Advanced Micro Devices (AMD) popped on Thursday. Piper Sandler analyst Harsh Kumar named the stock as a top pick in the large-cap space. He cites a number of factors, including the competitive positioning of its MI products, and says that he sees AMD ""as having bright prospects moving into the back half of the year."" Kumar has an Overweight rating and $175 price target on the stock. Yahoo Finance's Josh Lipton and Julie Hyman discuss the call in the video above. For more expert insight and the latest market action, click here to watch this full episode of Market Domination. This post was written by Stephanie Mikulich.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.99999976
2024-06-20,AMD Shares Now Top Piper Pick Because of AI Server Chips,Shares of Advanced Micro Devices traded higher Thursday after a Pipe Sandler analyst issued a bullish note on the artificial intelligence business of the chip maker.,130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.9999987
2024-06-20,Why Are AMD (AMD) Shares Soaring Today,"Shares of computer processor maker AMD (NASDAQ:AMD) jumped 7.5% in the morning session after Piper Sandler analyst Harsh Kumar named the company a ""Top Pick"" and maintained an Overweight (Buy) rating on the stock.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,1.0
2024-06-20,AMD Stock Is Set Up For Another Spike,AMD's upcoming quarterly earnings report on August 6th could serve as a catalyst for a potential spike in stock price. Find out if AMD stock is a buy.,130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.9999999
2024-06-20,Piper Sandler names AMD as top large-cap pick into 2H24,"Piper Sandler analysts named AMD (NASDAQ: NASDAQ:AMD) their top large-cap pick for the second half of 2024, citing positive feedback from their discussions with the chipmaker's management in Europe last week.",130.74778747558594,140.72532511567726,129.48810330722768,139.76557010788932,517768400,0.0645207730726693,Positive,0.9999999
2024-06-20,"AMD Hack Won’t Have a Material Impact on Business, Company Says","(Bloomberg) -- Advanced Micro Devices Inc. found hackers made off with limited information during a recent cyberattack, saying the infiltration shouldn’t have significant impact on its operations.Most Read from BloombergCar Dealerships Across US Halt Services After CyberattackPutin’s Hybrid War Opens a Second Front on NATO’s Eastern BorderHedge Fund Talent Schools Are Looking for the Perfect TraderCar Dealers Across US Are Crippled by a Second CyberattackWhat to Know About the Deadly Flesh-Eatin",130.74778747558594,140.725325115677

In [0]:
# df_with_sentiment.write.mode("overwrite").option("header", True).csv("dbfs:/FileStore/")